## Optimizing an Euler diagram of European countries and supranational organizations

See https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Supranational_European_Bodies.svg/960px-Supranational_European_Bodies.svg.png

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from vizopt.base import OptimConfig
from vizopt.schedules import make_term_schedules
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles_from_graph,
)

In [ ]:
with open(Path("../data/supranational_orgs.json")) as f:
    data = json.load(f)

data = [e for e in data if e["countryLabel"] != "Kingdom of the Netherlands"]

In [ ]:

# Build inclusion graph: org → country (parent → child)
# Leaf nodes = countries, internal nodes = orgs
G = nx.DiGraph()
coord_factor = 1/10

# Scale radii so area ∝ population (r ∝ sqrt(pop)), median radius = 0.2
sqrt_pops = np.sqrt([entry["population"] for entry in data])
r_scale = 0.2 / np.median(sqrt_pops)

for entry in data:
    country = entry["countryLabel"]
    lon, lat = entry["lon"], entry["lat"]
    r = r_scale * np.sqrt(entry["population"])
    r = max(min(r, 0.5), 0.01)

    # Deduplicate (Germany has duplicate entries in the raw data)
    org_labels = list(dict.fromkeys(entry["orgLabels"]))

    if not G.has_node(country):
        G.add_node(country, center=[lon*coord_factor, lat*coord_factor], r=r)

    for org in org_labels:
        if not G.has_node(org):
            G.add_node(org)
        G.add_edge(org, country)

org_names = [n for n in nx.topological_sort(G) if G.out_degree(n) > 0]
country_names = [n for n in G.nodes if G.out_degree(n) == 0]
print(f"{len(country_names)} countries, {len(org_names)} organizations")
print("Organizations:", org_names)


In [ ]:
n_iters = 20000
term_schedules = make_term_schedules(
    {
        "collision_delay": 0.27,
        "collision_ramp": 0.33,
        "exclusion_delay": 0.32,
        "exclusion_ramp": 0.47,
        "area_delay": 0.31,
        "area_ramp": 0.27,
        "perimeter_delay": 0.69,
        "perimeter_ramp": 0.45,
        "attraction_peak": 0.69,
        "attraction_ramp": 0.18,
    },
    n_iters,
)

named_results, named_circles_out, history, problem = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.0005,
        weight_circle_collision=100.0,
        weight_set_attraction=0.5,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

In [ ]:
import pandas as pd

df = pd.DataFrame(history).set_index("iteration")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

df[["total"]].plot(ax=axes[0], color="black", linewidth=1.5)
axes[0].set_ylabel("Loss")
axes[0].set_title("Total loss")
axes[0].legend()

component_cols = [c for c in df.columns if c != "total"]
df[component_cols].plot(ax=axes[1], linewidth=1.2)
axes[1].set_ylabel("Loss")
axes[1].set_title("Per-component loss")
axes[1].set_xlabel("Iteration")
axes[1].legend(fontsize=8)

plt.tight_layout()

In [ ]:
org_colors = dict(zip(org_names, plt.colormaps["tab10"](np.linspace(0, 0.9, len(org_names)))))

fig, ax = plt.subplots(figsize=(12, 8))

for org in org_names:
    result = named_results[org]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    color = org_colors[org]
    ax.fill(bx, by, alpha=0.12, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=org)

for country in country_names:
    cx_out, cy_out, _ = named_circles_out[country]
    r_orig = G.nodes[country]["r"]
    ax.add_patch(
        plt.Circle(
            (cx_out, cy_out),
            r_orig,
            facecolor="lightyellow",
            alpha=0.9,
            edgecolor="dimgray",
            linewidth=1.0,
        )
    )
    ax.text(cx_out, cy_out, country, ha="center", va="center", fontsize=8)

ax.set_aspect("equal")
ax.autoscale_view()
#ax.margins(0.05)
ax.legend(loc="upper left", fontsize=10, framealpha=0.8)
ax.set_title("European Supranational Organizations", fontsize=14)
ax.axis("off")
plt.tight_layout()


In [ ]:


df = pd.DataFrame(history).set_index("iteration")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

df[["total"]].plot(ax=axes[0], color="black", linewidth=1.5)
axes[0].set_ylabel("Loss")
axes[0].set_title("Total loss")
axes[0].legend()

component_cols = [c for c in df.columns if c != "total"]
df[component_cols].plot(ax=axes[1], linewidth=1.2)
axes[1].set_ylabel("Loss")
axes[1].set_title("Per-component loss")
axes[1].set_xlabel("Iteration")
axes[1].legend(fontsize=8)
axes[1].set_yscale("log")
plt.tight_layout()

In [ ]:
history[-1]

## With rectangles

In [ ]:
from vizopt.templates.euler.stars_vs_rectangles import (
    optimize_radially_convex_sets_and_rectangles_from_graph,
)

G_rect = nx.DiGraph()
coord_factor = 1 / 10

sqrt_pops = np.sqrt([entry["population"] for entry in data])
r_scale = 0.2 / np.median(sqrt_pops)

for entry in data:
    country = entry["countryLabel"]
    lon, lat = entry["lon"], entry["lat"]
    hw = max(min(r_scale * np.sqrt(entry["population"]), 0.5), 0.01)

    org_labels = list(dict.fromkeys(entry["orgLabels"]))
    if not G_rect.has_node(country):
        G_rect.add_node(country, center=[lon * coord_factor, lat * coord_factor], hw=hw, hh=hw)
    for org in org_labels:
        if not G_rect.has_node(org):
            G_rect.add_node(org)
        G_rect.add_edge(org, country)

org_names_rect = [n for n in nx.topological_sort(G_rect) if G_rect.out_degree(n) > 0]
country_names_rect = [n for n in G_rect.nodes if G_rect.out_degree(n) == 0]
print(f"{len(country_names_rect)} countries, {len(org_names_rect)} organizations")
print("Organizations:", org_names_rect)


In [ ]:
n_iters_rect = 20000
term_schedules_rect = make_term_schedules(
    {
        "rect_collision_delay": 0.27,
        "rect_collision_ramp": 0.33,
        "exclusion_delay": 0.32,
        "exclusion_ramp": 0.47,
        "area_delay": 0.31,
        "area_ramp": 0.27,
        "perimeter_delay": 0.69,
        "perimeter_ramp": 0.45,
        "set_attraction_peak": 0.69,
        "set_attraction_ramp": 0.18,
    },
    n_iters_rect,
)

named_results_rect, named_rects_out, history_rect, problem_rect = (
    optimize_radially_convex_sets_and_rectangles_from_graph(
        G_rect,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_convexity=10.0,
        weight_position_anchor=0.0005,
        weight_rect_collision=100.0,
        weight_set_attraction=0.5,
        term_schedules=term_schedules_rect,
        optim_config=OptimConfig(n_iters=n_iters_rect, learning_rate=0.002),
    )
)


In [ ]:
org_colors_rect = dict(zip(org_names_rect, plt.colormaps["tab10"](np.linspace(0, 0.9, len(org_names_rect)))))

fig, ax = plt.subplots(figsize=(12, 8))

for org in org_names_rect:
    result = named_results_rect[org]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    color = org_colors_rect[org]
    ax.fill(bx, by, alpha=0.12, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=org)

for country in country_names_rect:
    cx, cy, hw, hh = named_rects_out[country]
    ax.add_patch(
        plt.Rectangle(
            (cx - hw, cy - hh), 2 * hw, 2 * hh,
            facecolor="lightyellow",
            alpha=0.9,
            edgecolor="dimgray",
            linewidth=1.0,
        )
    )
    ax.text(cx, cy, country, ha="center", va="center", fontsize=8)

ax.set_aspect("equal")
ax.autoscale_view()
ax.legend(loc="upper left", fontsize=10, framealpha=0.8)
ax.set_title("European Supranational Organizations (Rectangles)", fontsize=14)
ax.axis("off")
plt.tight_layout()
